<a href="https://colab.research.google.com/github/camiloceron0304-prog/Taller_Semana6_Camilo_Ceron.ipynb-./blob/main/Copia_de_Taller_Semana7_Camilo_Ceron_completo_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller de Funciones, Modularización y Refactorización
### Dataset de Trabajo: Población de Bogotá (2005 - 2035)
1. Datos generales

Proyecto integrador: Dataset de Trabajo

Curso: Lógica Computacional

Programa: Ingeniería de Datos e Inteligencia Artificial

Semana: 7 de 14

Modalidad: Trabajo individual

2. Objetivo

Importante — Dataset de Trabajo: durante toda la Semana 7 debes trabajar con el dataset que cada estudiante está desarrollando. Los ejemplos de la guía son independientes y sirven para comprender el concepto; cuando se indique integración, adapta el código a las columnas, tipos de datos, reglas y operaciones de tu propio Dataset de Trabajo.

Aplicar funciones, parámetros, argumentos, valores de retorno, modularización y refactorización para mejorar la estructura interna de Dataset de Trabajo sin cambiar su comportamiento esperado.

Los ejercicios avanzan desde funciones sencillas hasta la refactorización del proyecto integrador.


3. Recomendaciones de trabajo
Antes de comenzar:

git status
git pull
Trabaja sobre la versión funcional de Dataset de Trabajo — versión de la Semana 6.

Antes de realizar cambios estructurales, crea una línea base:

git add .
git commit -m "chore: establece linea base antes de refactorizar"

**Curso:** Lógica Computacional · **Semana 7** ·

## Mi dataset

Trabajé con los 3 archivos:

| Archivo | Qué tiene |
|---|---|
| `osb_demografia-poblacion-localidad` | Población por año, localidad, sexo y edad |
| `osb_demografia-poblacion-upl` | Población por año, UPL, sexo y edad |
| `metadato-osb_demografia-poblacion` | Explicación de las columnas |

### Cómo adapté el taller

- Cada "valor" a clasificar es la **población total de una localidad o UPL en 2025**.
- En el archivo de localidades hay una fila llamada `Bogotá` (código 0) que es el total de la ciudad. La dejé por fuera para no sumar dos veces.
- Reglas de `clasificar_valor()`:

| Categoría | Localidades | UPL |
|---|---|---|
| `ALTO` | 500.000 o más | 300.000 o más |
| `NORMAL` | entre los dos límites | entre los dos límites |
| `BAJO` | menos de 150.000 | menos de 150.000 |



In [2]:
import csv
import glob
import io
from contextlib import redirect_stdout

ANIO_ANALISIS = 2025

# (limite_alto, limite_bajo) para cada dataset
LIMITES = {
    "localidad": (500_000, 150_000),
    "upl": (300_000, 150_000),
}

# Si estoy en Colab y los archivos no están, los subo
if not glob.glob("*poblacion-localidad*.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        pass


def buscar_archivo(patron):
    """Busca el archivo en la carpeta actual o en una carpeta datos."""
    for carpeta in [".", "datos", "..", "../datos"]:
        encontrados = glob.glob(carpeta + "/" + patron)
        if encontrados:
            return encontrados[0]
    raise FileNotFoundError("No encontré " + patron + ", copia los CSV junto al notebook")


RUTA_LOCALIDAD = buscar_archivo("*poblacion-localidad*.csv")
RUTA_UPL = buscar_archivo("*poblacion-upl*.csv")
RUTA_METADATO = buscar_archivo("metadato*.csv")


def cargar_csv(ruta):
    """Lee un CSV separado por ; y devuelve una lista de diccionarios."""
    try:
        with open(ruta, encoding="utf-8-sig", newline="") as archivo:
            return list(csv.DictReader(archivo, delimiter=";"))
    except UnicodeDecodeError:
        # el metadato viene en latin-1
        with open(ruta, encoding="latin-1", newline="") as archivo:
            return list(csv.DictReader(archivo, delimiter=";"))


datos = {
    "localidad": cargar_csv(RUTA_LOCALIDAD),
    "upl": cargar_csv(RUTA_UPL),
}
metadato = cargar_csv(RUTA_METADATO)

print("Filas en localidad:", len(datos["localidad"]))
print("Filas en UPL:", len(datos["upl"]))
print("Filas en metadato:", len(metadato))
print()
print("Primer registro de localidad:")
print(datos["localidad"][0])
print()
print("Diccionario de datos (4 primeras filas):")
for fila in metadato[:4]:
    print("-", fila["METADATO"].strip(), ":", fila["DESCRIPCIÓN"].strip())

Filas en localidad: 131502
Filas en UPL: 175956
Filas en metadato: 12

Primer registro de localidad:
{'ANO': '2005', 'CODIGO_LOCALIDAD': '0', 'NOMBRE_LOCALIDAD': 'Bogotá', 'SEXO': 'Hombres', 'EDAD': '6', 'CURSODEVIDA': 'Infancia', 'GRUPOEDAD': '00 a 11', 'POBLACION': '67184'}

Diccionario de datos (4 primeras filas):
- Serie de tiempo desde : 2005 - 2035
- Definición del indicador : Número de habitantes según características de interés
- Utilidad : Describir la distribución anual de la población según sexo, edad localidad y UPL de residencia
- Unidad de medición : Número de habitantes


## Antes de empezar (Git)

```bash
git status
git pull
git add .
git commit -m "chore: establece linea base antes de refactorizar"
```

---

# Reto 1: Primera función

In [7]:
def mostrar_mensaje():
    """Muestra un mensaje del dataset."""
    print("Procesando Dataset de Trabajo")


mostrar_mensaje()

Procesando Dataset de Trabajo


**Respuestas**

1. La definición es `def mostrar_mensaje(): ...`, ahí se crea la función.
2. La invocación es `mostrar_mensaje()`, ahí se ejecuta.
3. Su responsabilidad es mostrar un mensaje en pantalla.

---

# Reto 2: Parámetro y argumento

In [11]:
def saludar(nombre):
    """Saluda al estudiante que recibe."""
    print("Hola", nombre, "- bienvenido al Dataset de Trabajo")


saludar("Ana")
saludar("Carlos")
saludar("María")

Hola Ana - bienvenido al Dataset de Trabajo
Hola Carlos - bienvenido al Dataset de Trabajo
Hola María - bienvenido al Dataset de Trabajo


**Respuestas**

1. El parámetro es `nombre`.
2. Los argumentos son `"Ana"`, `"Carlos"` y `"María"`.
3. Con un parámetro la misma función sirve para cualquier nombre, sin repetir código.

---

# Reto 3: Función con retorno

In [14]:
def duplicar(valor):
    """Devuelve el doble del número."""
    return valor * 2


print(duplicar(5))
print(duplicar(0))
print(duplicar(10))

10
0
20


**Respuestas**

1. Recibe un número.
2. Calcula el doble.
3. Retorna ese resultado.
4. No queda guardado en ningún lado solo con el `return`. Hay que guardarlo en una variable (`doble = duplicar(5)`) o usarlo enseguida, por ejemplo dentro de un `print`.

---

# Reto 4: `print()` vs `return`

In [ ]:
def mostrar_resultado(valor):
    """Solo muestra el valor."""
    print(valor)


def obtener_resultado(valor):
    """Devuelve el valor para seguir usándolo."""
    return valor


a = mostrar_resultado(7)
b = obtener_resultado(7)

print("mostrar_resultado devolvió:", a)
print("obtener_resultado + 1 =", b + 1)

**Explicación:** `mostrar_resultado` imprime el 7 pero devuelve `None`, entonces no puedo seguir calculando con él. `obtener_resultado` **sí deja que otra parte del programa use el resultado**, por eso pude sumarle 1.

---

# Reto 5: Función de clasificación

`ALTO` si la población llega al límite alto, `BAJO` si está por debajo del límite bajo y `NORMAL` si queda en medio. Los límites van como parámetros porque localidades y UPL tienen tamaños distintos.

In [ ]:
def clasificar_valor(valor, limite_alto, limite_bajo):
    """Devuelve ALTO, NORMAL o BAJO según el valor."""
    if valor >= limite_alto:
        clasificacion = "ALTO"
    elif valor < limite_bajo:
        clasificacion = "BAJO"
    else:
        clasificacion = "NORMAL"
    return clasificacion


limite_alto, limite_bajo = LIMITES["localidad"]

print("Suba (1.251.652):", clasificar_valor(1_251_652, limite_alto, limite_bajo))
print("Puente Aranda (249.519):", clasificar_valor(249_519, limite_alto, limite_bajo))
print("La Candelaria (16.574):", clasificar_valor(16_574, limite_alto, limite_bajo))

**Pregunta:** ¿Por qué es mejor reutilizar `clasificar_valor()`?

Porque la regla queda escrita en un solo lugar. Si cambia un límite lo arreglo una vez y no tengo que buscar el mismo `if` por todo el código.

---

# Reto 6: Contador mediante una función

In [ ]:
def contar_registros(valores):
    """Cuenta cuántos valores hay en la lista."""
    contador = 0
    for valor in valores:
        contador += 1
    return contador


valores = [22, 55, 72, 38, 61]
print(contar_registros(valores))

**Pregunta:** ¿Por qué el contador se inicia dentro de la función?

Para que en cada llamada arranque en 0. Si estuviera afuera, la segunda llamada seguiría sumando desde donde quedó la primera y daría un número mal.

---

# Reto 7: Acumulador mediante una función

In [ ]:
def calcular_suma(valores):
    """Suma todos los valores de la lista."""
    suma = 0
    for valor in valores:
        suma += valor
    return suma


valores = [10, 20, 30, 40]
print(calcular_suma(valores))

La suma es una variable local, no uso ninguna variable global.

---

# Reto 8: Función con clasificación y resumen

In [ ]:
def contar_categorias(valores, limite_alto, limite_bajo):
    """Cuenta cuántos valores son ALTO, NORMAL y BAJO."""
    altos = 0
    normales = 0
    bajos = 0

    for valor in valores:
        categoria = clasificar_valor(valor, limite_alto, limite_bajo)
        if categoria == "ALTO":
            altos += 1
        elif categoria == "NORMAL":
            normales += 1
        else:
            bajos += 1

    return altos, normales, bajos


# Primero con la lista del taller (límites de ejemplo 70 y 40)
valores = [20, 55, 80, 40, 70]
altos, normales, bajos = contar_categorias(valores, 70, 40)
print("Lista del taller -> ALTO:", altos, "| NORMAL:", normales, "| BAJO:", bajos)


# Ahora con datos reales
def totalizar_por_territorio(registros, anio, campo_nombre):
    """Suma la población de cada territorio en un año. Devuelve {nombre: total}."""
    totales = {}
    for fila in registros:
        nombre = fila[campo_nombre]
        if int(fila["ANO"]) != anio:
            continue
        if nombre == "Bogotá":  # es el total de la ciudad, no una localidad
            continue
        if nombre not in totales:
            totales[nombre] = 0
        totales[nombre] += int(fila["POBLACION"])
    return totales


poblacion_localidades = totalizar_por_territorio(datos["localidad"], ANIO_ANALISIS, "NOMBRE_LOCALIDAD")
limite_alto, limite_bajo = LIMITES["localidad"]
altos, normales, bajos = contar_categorias(poblacion_localidades.values(), limite_alto, limite_bajo)
print("Localidades 2025 -> ALTO:", altos, "| NORMAL:", normales, "| BAJO:", bajos)

Cumple con parámetros, contadores, reutiliza `clasificar_valor()` y retorna los resultados.

---

# Reto 9: Separar el menú

In [ ]:
OPCION_SALIR = "3"


def mostrar_menu():
    """Muestra las opciones del programa."""
    print()
    print("===== DATASET DE TRABAJO: POBLACIÓN DE BOGOTÁ =====")
    print("1. Resumen por localidad")
    print("2. Resumen por UPL")
    print("3. Salir")


mostrar_menu()

**Pregunta:** ¿Qué responsabilidad queda en `main()`?

Controlar el flujo del programa: repetir el ciclo, pedir la opción, saber cuándo salir y llamar a las otras funciones. Ya no dibuja el menú ni calcula nada.

---

# Reto 10: Leer una opción

Para poder probarla sin teclear, la función recibe cómo pedir el dato (por defecto `input`).

In [ ]:
def leer_opcion(pedir=input):
    """Pide una opción y la devuelve sin espacios."""
    opcion = pedir("Seleccione una opción: ")
    return opcion.strip()


def simular_entradas(respuestas):
    """Reemplaza a input() con respuestas de una lista (para probar)."""
    respuestas = iter(respuestas)

    def pedir(mensaje):
        respuesta = next(respuestas)
        print(mensaje + respuesta)
        return respuesta

    return pedir


opcion = leer_opcion(simular_entradas(["  1 "]))
print("Opción leída:", repr(opcion))

**Respuestas**

1. Recibe la forma de pedir el dato (`pedir`), que normalmente es `input`.
2. Produce el texto de la opción elegida.
3. Uso `return` para que `main()` pueda guardar la opción y decidir qué hacer. Con `print` solo se vería en pantalla.

---

# Reto 11: Separar `ejecutar_opcion()`

`ejecutar_opcion()` necesita `procesar_registros()` (Reto 12), así que primero hago esa.

## Reto 12: Crear `procesar_registros()`

In [ ]:
def procesar_registros(territorios, limite_alto, limite_bajo, cantidad=None):
    """Procesa la población de los territorios y devuelve un diccionario con los resultados."""
    valores = list(territorios.values())
    if cantidad is not None:
        valores = valores[:cantidad]  # solo los primeros N

    altos, normales, bajos = contar_categorias(valores, limite_alto, limite_bajo)

    resultados = {
        "total": contar_registros(valores),
        "altos": altos,
        "normales": normales,
        "bajos": bajos,
        "suma": calcular_suma(valores),
        "existe_alto": altos > 0,  # bandera: hay al menos un ALTO
    }
    return resultados


limite_alto, limite_bajo = LIMITES["localidad"]
print(procesar_registros(poblacion_localidades, limite_alto, limite_bajo))
print("Solo 5:", procesar_registros(poblacion_localidades, limite_alto, limite_bajo, cantidad=5))

**Pregunta:** ¿Qué necesita recibir y qué debe retornar?

- **Recibe:** los datos (territorios con su población), los límites de clasificación y, si se quiere, la cantidad.
- **Retorna:** el resultado completo (total, contadores, suma y bandera). No imprime nada, de eso se encarga otra función.

## Reto 11: `ejecutar_opcion()`

In [ ]:
OPCIONES = {
    "1": {"nombre": "Localidades", "dataset": "localidad", "campo": "NOMBRE_LOCALIDAD"},
    "2": {"nombre": "UPL", "dataset": "upl", "campo": "NOMBRE_UPL"},
}


def ejecutar_opcion(opcion, datos, anio=ANIO_ANALISIS):
    """Hace el análisis de la opción elegida. Devuelve None si la opción no es válida."""
    if opcion not in OPCIONES:
        print("Opción inválida. Elija 1, 2 o 3.")
        return None

    config = OPCIONES[opcion]
    registros = datos[config["dataset"]]
    limite_alto, limite_bajo = LIMITES[config["dataset"]]

    totales = totalizar_por_territorio(registros, anio, config["campo"])
    resultados = procesar_registros(totales, limite_alto, limite_bajo)
    resultados["titulo"] = "Resumen - " + config["nombre"] + " " + str(anio)
    return resultados


print(ejecutar_opcion("2", datos))
print(ejecutar_opcion("9", datos))

`ejecutar_opcion()` solo coordina: elige el dataset y llama a `totalizar_por_territorio()` y `procesar_registros()`. No tiene la lógica de cada paso.

---

## Reto 12: Crear `procesar_registros()`

In [ ]:
def procesar_registros(territorios, limite_alto, limite_bajo, cantidad=None):
    """
    Procesa la población de los territorios y retorna un diccionario de resultados.

    territorios: diccionario {nombre: población}.
    cantidad: si se indica, solo procesa los primeros N territorios.
    """
    valores = list(territorios.values())
    if cantidad is not None:
        valores = valores[:cantidad]

    altos, normales, bajos = contar_categorias(valores, limite_alto, limite_bajo)

    return {
        "total": contar_registros(valores),
        "altos": altos,
        "normales": normales,
        "bajos": bajos,
        "suma": calcular_suma(valores),
        "existe_alto": altos > 0,  # bandera: ¿hubo al menos un ALTO?
    }


resultado = procesar_registros(poblacion_localidades, *LIMITES["localidad"])
print(resultado)
print("Solo los 5 primeros:", procesar_registros(poblacion_localidades, *LIMITES["localidad"], cantidad=5))

# Reto 13: Función de resumen

In [ ]:
def mostrar_resumen(total, altos, normales, bajos, suma, existe_alto, titulo="Resumen"):
    """Muestra el resumen. No calcula nada."""
    print("===", titulo, "===")
    print("Territorios procesados :", total)
    print("ALTO                   :", altos)
    print("NORMAL                 :", normales)
    print("BAJO                   :", bajos)
    print("Población total        :", f"{suma:,}")
    if existe_alto:
        print("¿Existe algún ALTO?    : Sí")
    else:
        print("¿Existe algún ALTO?    : No")


r = ejecutar_opcion("1", datos)
mostrar_resumen(r["total"], r["altos"], r["normales"], r["bajos"], r["suma"], r["existe_alto"], r["titulo"])
print()
r = ejecutar_opcion("2", datos)
mostrar_resumen(r["total"], r["altos"], r["normales"], r["bajos"], r["suma"], r["existe_alto"], r["titulo"])

**Pregunta:** ¿Por qué separar el cálculo de la presentación?

Porque así puedo probar el cálculo sin depender de lo que sale en pantalla, y si después quiero mostrar el resultado de otra forma (un archivo, un gráfico) no tengo que tocar los cálculos.

---

# Reto 14: Reducir la lógica de `main()`

In [ ]:
def main(datos, pedir=input):
    """Coordina el programa: menú, opción, análisis y resumen."""
    while True:
        mostrar_menu()
        opcion = leer_opcion(pedir)

        if opcion == OPCION_SALIR:
            print("Hasta luego.")
            break

        r = ejecutar_opcion(opcion, datos)
        if r is not None:
            mostrar_resumen(r["total"], r["altos"], r["normales"], r["bajos"],
                            r["suma"], r["existe_alto"], r["titulo"])


# Prueba: opción 1, opción 2, una inválida y salir
main(datos, simular_entradas(["1", "2", "9", "3"]))

# Para usarlo de verdad con el teclado, quitar el # de la siguiente línea:
# main(datos)

**Estructura que quedó**

```text
main()
 ├── mostrar_menu()
 ├── leer_opcion()
 ├── ejecutar_opcion()
 │     ├── totalizar_por_territorio()
 │     └── procesar_registros()
 │            ├── contar_registros()
 │            ├── calcular_suma()
 │            └── contar_categorias()
 │                   └── clasificar_valor()
 └── mostrar_resumen()
```

**Antes**, `main()` tenía todo mezclado: menú, entrada, recorrido de datos, clasificación, contadores y prints.

**Pregunta:** ¿Qué responsabilidades quedaron fuera de `main()`?

El menú, la lectura de la opción, la selección del análisis, el procesamiento, la clasificación y el resumen. A `main()` solo le quedó el ciclo y la decisión de salir.

---

# Reto 15: Detectar variables globales

Para comparar necesito la **línea base**: el análisis de localidades 2025 hecho todo seguido, sin funciones y con variables globales. Guardo sus resultados para el Reto 17.

In [ ]:
# ---------- LÍNEA BASE (antes de refactorizar) ----------
totales_localidad = {}
for fila in datos["localidad"]:
    if int(fila["ANO"]) == 2025 and fila["NOMBRE_LOCALIDAD"] != "Bogotá":
        nombre_loc = fila["NOMBRE_LOCALIDAD"]
        if nombre_loc not in totales_localidad:
            totales_localidad[nombre_loc] = 0
        totales_localidad[nombre_loc] += int(fila["POBLACION"])

contador_total = 0
contador_altos = 0
contador_normales = 0
contador_bajos = 0
suma_poblacion = 0
existe_alto = False
clasificacion_antes = {}

for nombre_loc in totales_localidad:
    poblacion = totales_localidad[nombre_loc]
    contador_total += 1
    suma_poblacion += poblacion

    if poblacion >= 500_000:
        categoria = "ALTO"
        contador_altos += 1
        existe_alto = True
    elif poblacion < 150_000:
        categoria = "BAJO"
        contador_bajos += 1
    else:
        categoria = "NORMAL"
        contador_normales += 1

    clasificacion_antes[nombre_loc] = categoria

print("=== Resumen - Localidades 2025 ===")
print("Territorios procesados :", contador_total)
print("ALTO                   :", contador_altos)
print("NORMAL                 :", contador_normales)
print("BAJO                   :", contador_bajos)
print("Población total        :", f"{suma_poblacion:,}")
print("¿Existe algún ALTO?    :", "Sí" if existe_alto else "No")

### Variables globales de la línea base

| Variable | ¿Es global? | ¿Puede ser parámetro? | ¿Puede retornarse? |
|---|---|---|---|
| `totales_localidad` | Sí | Sí, se la paso a `procesar_registros` | Sí, la devuelve `totalizar_por_territorio` |
| `contador_total` | Sí | No, empieza en 0 dentro de la función | Sí, con `contar_registros` |
| `contador_altos`, `contador_normales`, `contador_bajos` | Sí | No, empiezan en 0 dentro de la función | Sí, `contar_categorias` los retorna |
| `suma_poblacion` | Sí | No, es un acumulador local | Sí, con `calcular_suma` |
| `existe_alto` | Sí | No | Sí, va en el resultado |
| `500_000` y `150_000` dentro del `if` | Valores sueltos | Sí (`limite_alto`, `limite_bajo`) | No aplica |

Todas eran dependencias globales innecesarias. En la versión modular la información viaja por parámetros y retornos. Solo dejé como globales las constantes de configuración (`LIMITES`, `OPCIONES`, `ANIO_ANALISIS`), que no cambian.

---

# Reto 16: Refactorizar sin cambiar el comportamiento

```text
1. Ejecutar el dataset              -> línea base del Reto 15
2. Registrar resultados             -> contadores y clasificacion_antes
3. Commit de línea base             -> ver Git al final
4. Identificar responsabilidades    -> totalizar, clasificar, contar, sumar, mostrar
5. Crear funciones                  -> Retos 5 a 14
6. Pasar información por parámetros -> límites, valores, territorios
7. Retornar resultados              -> return en las funciones de cálculo
8. Ejecutar de nuevo                -> celda de abajo
9. Comparar resultados              -> Reto 17
```

In [ ]:
# ---------- DESPUÉS de refactorizar ----------
resultados_despues = ejecutar_opcion("1", datos)
r = resultados_despues
mostrar_resumen(r["total"], r["altos"], r["normales"], r["bajos"], r["suma"], r["existe_alto"], r["titulo"])

---

# Reto 17: Pruebas antes y después

Comparo la **línea base** (Reto 15, todo seguido y con globales) contra la **versión modular** (Reto 16).
Cada prueba muestra el resultado de *antes*, el de *después* y si coinciden.
Las pruebas 1 a 4 usan los resultados reales de la línea base. En la 5 y la 6 el "antes" es el comportamiento
que ya tenía definido el programa (mensaje de error y mensaje de salida).

In [ ]:
resultados_pruebas = []  # guarda True/False por cada comparación


def comparar(prueba, antes, despues):
    """ Compara el resultado de antes con el de después y lo muestra. """
    coincide = antes == despues
    resultados_pruebas.append(coincide)
    print(prueba)
    print("   Antes     :", antes)
    print("   Después   :", despues)
    print("   ¿Coincide?:", "Sí" if coincide else "NO")
    print()


# 1. Clasificación: la categoría de cada localidad
clasificacion_despues = {}
for nombre_loc, poblacion in poblacion_localidades.items():
    clasificacion_despues[nombre_loc] = clasificar_valor(poblacion, *LIMITES["localidad"])

comparar("1. Clasificación (categoría de cada localidad)",
         list(clasificacion_antes.values()),
         list(clasificacion_despues.values()))

# 2. Procesamiento: total, altos, normales, bajos, suma y bandera
r = resultados_despues
antes = (contador_total, contador_altos, contador_normales, contador_bajos, suma_poblacion, existe_alto)
despues = (r["total"], r["altos"], r["normales"], r["bajos"], r["suma"], r["existe_alto"])
comparar("2. Procesamiento (total, altos, normales, bajos, suma, existe_alto)", antes, despues)

# 3. Resumen: las líneas que se imprimen en pantalla
texto_antes = io.StringIO()
with redirect_stdout(texto_antes):  # redirect_stdout guarda lo que imprime el print en una variable
    print("===", "Resumen - Localidades 2025", "===")
    print("Territorios procesados :", contador_total)
    print("ALTO                   :", contador_altos)
    print("NORMAL                 :", contador_normales)
    print("BAJO                   :", contador_bajos)
    print("Población total        :", f"{suma_poblacion:,}")
    print("¿Existe algún ALTO?    :", "Sí" if existe_alto else "No")

texto_despues = io.StringIO()
with redirect_stdout(texto_despues):
    mostrar_resumen(r["total"], r["altos"], r["normales"], r["bajos"], r["suma"], r["existe_alto"], r["titulo"])

comparar("3. Resumen (líneas impresas)", texto_antes.getvalue().splitlines(), texto_despues.getvalue().splitlines())

# 4. Opción válida: la opción 1 debe dar las mismas localidades de la línea base
resultado_opcion1 = ejecutar_opcion("1", datos)
comparar("4. Opción válida (total de localidades)", contador_total, resultado_opcion1["total"])

# 5. Opción inválida: debe mostrar el mensaje y no devolver resultados
texto_error = io.StringIO()
with redirect_stdout(texto_error):
    resultado_invalido = ejecutar_opcion("9", datos)

comparar("5a. Opción inválida (mensaje)", "Opción inválida. Elija 1, 2 o 3.", texto_error.getvalue().strip())
comparar("5b. Opción inválida (resultado)", None, resultado_invalido)

# 6. Salida: el programa termina diciendo "Hasta luego."
texto_salida = io.StringIO()
with redirect_stdout(texto_salida):
    main(datos, simular_entradas(["3"]))

ultima_linea = texto_salida.getvalue().strip().splitlines()[-1]
comparar("6. Salida (último mensaje)", "Hasta luego.", ultima_linea)

print("¿Todas las pruebas coinciden?", all(resultados_pruebas))

**Conclusión:** si todas las filas dicen `Sí`, la refactorización no cambió el comportamiento: las mismas
localidades, las mismas categorías, la misma suma y el mismo resumen que la línea base.

---

# Reto 18: Pruebas de funciones

In [ ]:
pruebas_ok = []  # guarda True/False por cada prueba


def probar(funcion, entrada, esperado, obtenido):
    """ Muestra una prueba y guarda si salió bien. """
    ok = esperado == obtenido
    pruebas_ok.append(ok)
    print(funcion, "|", entrada, "| esperado:", esperado, "| obtenido:", obtenido, "|", "OK" if ok else "FALLA")


limite_alto, limite_bajo = LIMITES["localidad"]  # 500.000 y 150.000

# clasificar_valor con localidades reales
probar("clasificar_valor", "Suba 1.251.652", "ALTO", clasificar_valor(1_251_652, limite_alto, limite_bajo))
probar("clasificar_valor", "Puente Aranda 249.519", "NORMAL", clasificar_valor(249_519, limite_alto, limite_bajo))
probar("clasificar_valor", "La Candelaria 16.574", "BAJO", clasificar_valor(16_574, limite_alto, limite_bajo))

# clasificar_valor justo en los límites
probar("clasificar_valor", "500.000 (límite alto)", "ALTO", clasificar_valor(500_000, limite_alto, limite_bajo))
probar("clasificar_valor", "150.000 (límite bajo)", "NORMAL", clasificar_valor(150_000, limite_alto, limite_bajo))
probar("clasificar_valor", "149.999", "BAJO", clasificar_valor(149_999, limite_alto, limite_bajo))

# funciones sencillas de los primeros retos
probar("duplicar", "5", 10, duplicar(5))
probar("contar_registros", "[10, 20, 30]", 3, contar_registros([10, 20, 30]))
probar("calcular_suma", "[10, 20, 30, 40]", 100, calcular_suma([10, 20, 30, 40]))
probar("contar_categorias", "[20, 55, 80, 40, 70] con 70 y 40", (2, 2, 1),
       contar_categorias([20, 55, 80, 40, 70], 70, 40))

# totalizar_por_territorio con filas inventadas: suma solo 2025 y deja por fuera "Bogotá"
filas_prueba = [
    {"ANO": "2025", "NOMBRE_LOCALIDAD": "X", "POBLACION": "10"},
    {"ANO": "2025", "NOMBRE_LOCALIDAD": "X", "POBLACION": "5"},
    {"ANO": "2024", "NOMBRE_LOCALIDAD": "X", "POBLACION": "100"},
    {"ANO": "2025", "NOMBRE_LOCALIDAD": "Bogotá", "POBLACION": "999"},
]
probar("totalizar_por_territorio", "filas de X y de Bogotá", {"X": 15},
       totalizar_por_territorio(filas_prueba, 2025, "NOMBRE_LOCALIDAD"))

# procesar_registros con territorios inventados
esperado = {"total": 3, "altos": 1, "normales": 1, "bajos": 1, "suma": 900_000, "existe_alto": True}
obtenido = procesar_registros({"A": 600_000, "B": 200_000, "C": 100_000}, limite_alto, limite_bajo)
probar("procesar_registros", "A=600.000, B=200.000, C=100.000", esperado, obtenido)

obtenido = procesar_registros({"A": 10, "B": 20}, limite_alto, limite_bajo)
probar("procesar_registros (existe_alto)", "A=10, B=20", False, obtenido["existe_alto"])

# leer_opcion con una respuesta simulada (el eco que imprime se descarta)
with redirect_stdout(io.StringIO()):
    opcion_leida = leer_opcion(simular_entradas(["  2 "]))
probar("leer_opcion", "'  2 '", "2", opcion_leida)

print()
print("¿Todas las pruebas pasaron?", all(pruebas_ok))

Probé más de tres funciones, incluidos los **casos límite** (500.000 y 150.000 exactos), que son donde más fácil
se equivoca un `>=` o un `<`.

---

# Reto 19: Análisis de responsabilidad

| Función | ¿Qué recibe? | ¿Qué hace? | ¿Qué retorna? |
|---|---|---|---|
| `mostrar_menu()` | Nada | Imprime las opciones del programa | Nada (`None`) |
| `leer_opcion(pedir)` | La forma de pedir el dato (`input` por defecto) | Pide la opción y le quita los espacios | El texto de la opción |
| `clasificar_valor(valor, limite_alto, limite_bajo)` | Un valor y los dos límites | Compara el valor con los límites | `"ALTO"`, `"NORMAL"` o `"BAJO"` |
| `contar_registros(valores)` | Una lista de valores | Cuenta con un contador | La cantidad |
| `calcular_suma(valores)` | Una lista de valores | Acumula la suma | La suma |
| `contar_categorias(valores, limite_alto, limite_bajo)` | Valores y límites | Clasifica cada valor y cuenta cada categoría | `altos, normales, bajos` |
| `totalizar_por_territorio(registros, anio, campo_nombre)` | Filas del CSV, año y nombre de la columna | Suma la población de cada territorio en ese año (sin la fila `Bogotá`) | Diccionario `{nombre: total}` |
| `procesar_registros(territorios, limite_alto, limite_bajo, cantidad)` | Territorios con su población y los límites | Calcula total, contadores, suma y bandera | Diccionario de resultados |
| `ejecutar_opcion(opcion, datos, anio)` | La opción, los datos y el año | Valida la opción, elige el dataset, totaliza y procesa | Diccionario de resultados, o `None` si es inválida |
| `mostrar_resumen(...)` | Los resultados y un título | Imprime el resumen | Nada (`None`) |
| `main(datos, pedir)` | Los datos y la forma de pedir | Repite el ciclo: menú, opción, análisis y resumen | Nada (`None`) |

**Pregunta:** ¿Existe alguna función que esté realizando demasiadas responsabilidades?

La que más hace es `ejecutar_opcion()`: valida la opción, elige el dataset y los límites, llama a dos funciones
y además arma el título y imprime el mensaje de error. Todavía es corta y solo coordina, pero si el proyecto
crece conviene separar la validación y el mensaje. `procesar_registros()` también hace varias cosas, pero cada una
la delega en otra función (`contar_registros`, `calcular_suma`, `contar_categorias`), así que se entiende bien.

---

# Reto 20: Documentar las funciones

**`clasificar_valor(valor, limite_alto, limite_bajo)`**
Responsabilidad: recibir una población y retornar su categoría (`ALTO`, `NORMAL` o `BAJO`) según los límites.

**`totalizar_por_territorio(registros, anio, campo_nombre)`**
Responsabilidad: sumar la población de cada localidad o UPL en un año y retornar un diccionario `{nombre: total}`.
Deja por fuera la fila `Bogotá` para no contar dos veces.

**`procesar_registros(territorios, limite_alto, limite_bajo, cantidad)`**
Responsabilidad: procesar la población de los territorios y retornar los resultados (total, contadores, suma y bandera).

**`contar_categorias(valores, limite_alto, limite_bajo)`**
Responsabilidad: clasificar cada valor con `clasificar_valor()` y retornar cuántos son ALTO, NORMAL y BAJO.

**`contar_registros(valores)` y `calcular_suma(valores)`**
Responsabilidad: retornar la cantidad de valores y la suma de la lista, respectivamente.

**`ejecutar_opcion(opcion, datos, anio)`**
Responsabilidad: coordinar el análisis de la opción elegida y retornar sus resultados (o `None` si es inválida).

**`mostrar_resumen(total, altos, normales, bajos, suma, existe_alto, titulo)`**
Responsabilidad: presentar en pantalla el resumen generado por el procesamiento. No calcula nada.

**`mostrar_menu()` y `leer_opcion(pedir)`**
Responsabilidad: mostrar las opciones y retornar la opción elegida por el usuario.

**`main(datos, pedir)`**
Responsabilidad: coordinar el ciclo del programa hasta que el usuario elija salir.

---

# Reto final: Dataset de Trabajo, versión 0.7

El programa modular son las funciones de los retos 5 a 14. Se ejecuta con `main(datos)` (con teclado) o con
`main(datos, simular_entradas([...]))` (con respuestas simuladas, como en el Reto 14).

### Pseudocódigo actualizado

```text
INICIO
    cargar los CSV de localidad y UPL en "datos"
    REPETIR
        mostrar_menu()
        opcion = leer_opcion()
        SI opcion = "3" ENTONCES
            mostrar "Hasta luego."
            SALIR del ciclo
        FIN SI
        resultados = ejecutar_opcion(opcion, datos)
        SI resultados no es vacío ENTONCES
            mostrar_resumen(resultados)
        FIN SI
    FIN REPETIR
FIN

ejecutar_opcion(opcion, datos):
    SI opcion no es "1" ni "2": mostrar error y retornar vacío
    totales = totalizar_por_territorio(datos del dataset elegido, 2025)
    resultados = procesar_registros(totales, límites del dataset)
    retornar resultados

procesar_registros(territorios, límites):
    PARA CADA territorio: clasificar_valor(población, límites)
    contar categorías, sumar población y marcar si existe algún ALTO
    retornar resultados
```

### Verificación final

In [ ]:
print("Comparaciones antes/después:", len(resultados_pruebas), "-> todas coinciden:", all(resultados_pruebas))
print("Pruebas de funciones       :", len(pruebas_ok), "-> todas pasan:", all(pruebas_ok))
print()

if all(resultados_pruebas) and all(pruebas_ok):
    print("Versión 0.7 verificada")
else:
    print("Hay pruebas que fallan, revisar")

### Requisitos mínimos

```text
[x] main() actúa principalmente como coordinador.
[x] Existen funciones con responsabilidades claras.
[x] Se utilizan parámetros.
[x] Se utilizan argumentos.
[x] Se utiliza return.
[x] Se reutiliza clasificar_valor().
[x] Se separa el procesamiento del menú.
[x] Se reduce la duplicación.
[x] Se evitan variables globales innecesarias.
[x] Se mantienen las capacidades anteriores.
[x] Se realizan pruebas antes y después (Retos 17 y 18).
[x] Se actualiza la documentación (Reto 20 y pseudocódigo).
[ ] Se crea commit de línea base.
[ ] Se crea commit de refactorización.
[ ] Se realiza push a GitHub.
```

---

## Git: commits y push

Los commits y el push se hacen en la terminal (no se pueden hacer desde el notebook):

```bash
# línea base (ANTES de refactorizar, si aún no la hiciste)
git add .
git commit -m "chore: establece linea base antes de refactorizar"

# refactorización (con este notebook ya terminado)
git add .
git commit -m "refactor: modulariza Dataset de Trabajo en funciones (v0.7)"

git push
```

---

# Preguntas de reflexión

1. **¿Qué problema resuelven las funciones?** Permiten escribir una tarea una sola vez y usarla muchas veces. Por ejemplo, `clasificar_valor()` se usa para localidades y para UPL sin copiar el `if`.
2. **¿Diferencia entre parámetro y argumento?** El parámetro es el nombre que aparece en la definición (`nombre`, `limite_alto`). El argumento es el valor concreto que le paso al llamarla (`"Ana"`, `500_000`).
3. **¿Por qué `return` es importante?** Porque entrega el resultado para que otra parte del programa lo use. Sin `return`, `procesar_registros()` no podría pasarle los resultados a `mostrar_resumen()`.
4. **¿Cuándo conviene utilizar una función?** Cuando una tarea se repite, cuando tiene una responsabilidad clara o cuando quiero probarla sola.
5. **¿Diferencia entre `print()` y `return`?** `print()` solo muestra en pantalla y la función devuelve `None`. `return` devuelve el valor para seguir trabajando con él (en el Reto 4 pude sumarle 1 a `obtener_resultado`, pero no a `mostrar_resultado`).
6. **¿Ventaja de dividir el Dataset de Trabajo en funciones?** El código es más corto de leer, cada parte se prueba por separado y puedo cambiar una sin dañar las demás. Con 131.502 filas de localidades y 175.956 de UPL, es más fácil encontrar un error cuando cada paso está separado.
7. **¿Por qué no depender de variables globales?** Porque cualquier parte del programa puede cambiarlas sin que yo me dé cuenta, y las funciones dejan de ser independientes: dan resultados distintos según lo que se ejecutó antes.
8. **¿Qué significa refactorizar?** Reorganizar el código por dentro (en funciones, sin duplicación) sin cambiar lo que hace por fuera.
9. **¿Qué riesgo hay al refactorizar un programa que ya funciona?** Dañar algo que andaba bien sin notarlo, por ejemplo un `>=` que pasa a `>` y cambia una clasificación. Por eso comparé los resultados antes y después.
10. **¿Por qué fue útil crear un commit antes?** Porque deja una versión que funciona a la cual volver si la refactorización sale mal, y permite ver exactamente qué cambió.

---

# Checklist final

```text
[x] Puedo explicar qué es una función.
[x] Puedo diferenciar parámetro y argumento.
[x] Puedo explicar return.
[x] Puedo diferenciar print() y return.
[x] Puedo identificar responsabilidades.
[x] main() ya no contiene toda la lógica.
[x] Mi código tiene menos duplicación.
[x] Evité dependencias globales innecesarias.
[x] Probé Dataset de Trabajo antes de refactorizar.
[x] Probé Dataset de Trabajo después de refactorizar.
[x] Los resultados esperados se mantienen.
[x] Actualicé la documentación.
[ ] Creé el commit de línea base.
[ ] Creé el commit de refactorización.
[ ] Hice push al repositorio.
[x] Puedo explicar por qué dividí el código de esa manera.
```

### Producto de cierre

Una solución modularizada mediante funciones para procesar el Dataset de Trabajo: **población de Bogotá por
localidad y UPL (2005-2035)**. La siguiente semana esta modularización sirve de base para trabajar con
estructuras de datos.